# 15 — Upskilling Recommendation Engine

**Day 3, Step 15.** v1 rules, then v2 semantic — exactly as the Build Notes
stage it.

The catalogue is built **only from what is in the data**: the five real
`Training Program Name` values plus O*NET's own vocabulary, with real
`Training Cost` and duration attached. That is what makes the ROI engine in
notebook 17 priced rather than notional.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.skills.recommend import build_course_catalogue, recommend_v1, recommend_semantic

catalogue = build_course_catalogue()
catalogue[["course", "median_cost", "median_days", "observed_pass_rate",
           "times_delivered", "source"]]

2026-08-28 01:55:17 | INFO  | course catalogue built


,course,median_cost,median_days,observed_pass_rate,times_delivered,source
0,Communication Skills,542.94,3.0,0.540,705,observed
1,Customer Service,588.88,3.0,0.527,594,observed
2,Leadership Development,576.10,3.0,0.531,599,observed
3,Project Management,597.13,3.0,0.484,638,observed
4,Technical Skills,572.60,3.0,0.433,614,observed
5,Analytical Thinking and Problem Solving,576.10,3.0,0.503,0,derived_from_onet_vocabulary
6,Applied Scientific Method,576.10,3.0,0.503,0,derived_from_onet_vocabulary
7,Business Intelligence and Data Visualisation,576.10,3.0,0.503,0,derived_from_onet_vocabulary
8,Business Writing and Documentation,576.10,3.0,0.503,0,derived_from_onet_vocabulary
9,Cloud Platforms and Deployment,576.10,3.0,0.503,0,derived_from_onet_vocabulary


## v1 — rules

In [3]:
for skill in ["MLOps", "SQL", "Critical Thinking", "Generative AI"]:
    print(f"  {skill:22} -> {recommend_v1(skill)}")

  MLOps                  -> None
  SQL                    -> Data Fundamentals: SQL and Relational Databases
  Critical Thinking      -> Analytical Thinking and Problem Solving
  Generative AI          -> None


## v2 — semantic

A sentence-transformer embeds the missing skill and each course description, so
`MLOps` can reach *"Deploying and Monitoring Machine Learning Systems"* even
though the words do not overlap at all.

A rule always wins when one exists — it is exact and explainable. And below a
confidence floor the engine says so rather than presenting a confident-looking
wrong answer.

In [4]:
recommend_semantic(["SQL", "MLOps", "Microsoft SharePoint", "The MathWorks MATLAB",
                    "Kubernetes", "Zzyzx Quantum Basketweaving"])

2026-08-28 01:55:19 | INFO  | course catalogue built


2026-08-28 01:55:19 | INFO  | No device provided, using mps


2026-08-28 01:55:20 | INFO  | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-08-28 01:55:20 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-08-28 01:55:20 | INFO  | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


...

2026-08-28 01:55:31 | INFO  | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-28 01:55:31 | INFO  | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


2026-08-28 01:55:31 | INFO  | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


2026-08-28 01:55:31 | INFO  | recommendations generated


,skill,course,match_confidence,method,median_cost,median_days
0,SQL,Data Fundamentals: SQL and Relational Databases,1.0000,rule,576.1,3.0
1,MLOps,Cloud Platforms and Deployment,0.3198,semantic,576.1,3.0
2,Microsoft SharePoint,Enterprise Systems: CRM and Salesforce,0.3892,semantic,576.1,3.0
3,The MathWorks MATLAB,Statistical Programming with R,0.3763,semantic,576.1,3.0
4,Kubernetes,Containerisation and Deployment Pipelines,1.0000,rule,576.1,3.0
5,Zzyzx Quantum Basketweaving,Technical Skills,0.1333,fallback_low_confidence,572.6,3.0
